<a href="https://colab.research.google.com/github/MaGiaVy/DoAnPython/blob/main/project/notebooks/Nhom3thangcuti_Tuan4/notebooks/Tuan4_TwoStage_MobileCLIP_DINOv2_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Tuần 4 — 2-Stage Visual Search: MobileCLIP + YOLO-Crop DINOv2
## Shopee Product Matching | Google Colab T4 GPU

### Pipeline Architecture
```
Query Image + Title
     │
     ├─→ [YOLO Crop]  → Cropped product region
     │
     ├─→ [Stage 1: MobileCLIP]
     │   ├─ img_feat (original) + txt_feat
     │   ├─ fused = normalize(α·img + (1-α)·txt)
     │   └─ FAISS search → top-K candidates (K=100)
     │
     └─→ [Stage 2: DINOv2-Small on YOLO crop]
         ├─ dino_score = cosine(q_dino, gallery_dino[idx])
         ├─ fused_score = β·dino + (1-β)·clip
         └─ Return top-5
```

### Expected Results
| Metric | Baseline MobileCLIP | Target (2-Stage) | Gain |
|--------|---------------------|-------------------|------|
| mAP@5 | 0.7708 | ~0.79–0.80 | +2–3% |
| Precision@1 | 0.7937 | ~0.81–0.82 | +1–2% |
| Recall@5 | 0.7430 | ~0.76–0.77 | +1–2% |

### Notebook Sections
- **Cell 0**: Kiểm tra GPU
- **Cell 1**: Cài đặt thư viện
- **Cell 2**: Mount Drive & Load data
- **Cell 3**: Cấu hình & Chia tập
- **Cell 4**: Load MobileCLIP model
- **Cell 5**: Trích xuất MobileCLIP features (gallery)
- **Cell 6**: YOLO crop toàn bộ gallery
- **Cell 7**: Trích xuất DINOv2 features (YOLO crops)
- **Cell 8**: Grid search Alpha (Stage 1)
- **Cell 9**: Grid search Retrieval-K (Stage 2, β=1.0)
- **Cell 10**: Grid search Beta (full pipeline)
- **Cell 11**: Đánh giá Test set (chạy 1 lần!)
- **Cell 12**: So sánh với Baseline & Báo cáo
- **Cell 13**: Visualize kết quả truy vấn
- **Cell 14**: Demo single query

## ⚙️ Cell 0: Kiểm tra GPU & Runtime

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU khả dụng:')
    print(result.stdout)
else:
    print('❌ Không tìm thấy GPU!')
    print('👉 Vào Runtime > Change runtime type > chọn T4 GPU!')

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
    if vram < 14:
        print('⚠️  VRAM < 14 GB — hãy giảm BATCH_SIZE_DINO xuống 16')

## 📦 Cell 1: Cài đặt thư viện

In [ ]:
import sys, subprocess

# Core dependencies
!pip install -q faiss-cpu ultralytics timm tqdm

# ─── Thử cài MobileCLIP (Apple) ────────────────────────────────────────────
USE_MOBILECLIP = False
print('⏳ Đang thử cài MobileCLIP (Apple)...')
try:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'git+https://github.com/apple/ml-mobileclip.git'],
        capture_output=True, text=True, timeout=180
    )
    import mobileclip
    USE_MOBILECLIP = True
    print('✅ MobileCLIP (Apple) sẵn sàng!')
except Exception as e:
    print(f'⚠️  Không cài được MobileCLIP: {e}')
    print('🔄 Fallback → openai/clip-vit-base-patch32 (HuggingFace)')
    !pip install -q transformers

print(f'\n🔧 Chế độ CLIP: {"MobileCLIP (Apple)" if USE_MOBILECLIP else "CLIP HuggingFace"}')

## 📂 Cell 2: Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ─── Tự động dò tìm dataset ────────────────────────────────────────────────
POSSIBLE_PATHS = [
    '/content/drive/MyDrive/DoAnPython/DuLieuPython',
    '/content/drive/MyDrive/DuLieuPython',
    '/content/drive/My Drive/DoAnPython/DuLieuPython',
    '/content/drive/My Drive/DuLieuPython',
    '/content/drive/MyDrive/DuLieuPython/DuLieuPython',
]

DATA_DIR = None
for path in POSSIBLE_PATHS:
    if os.path.exists(os.path.join(path, 'train.csv')):
        DATA_DIR = path
        break

if DATA_DIR is None:
    DATA_DIR = '/content/drive/MyDrive/DuLieuPython/DuLieuPython'
    print(f'⚠️  Dùng đường dẫn mặc định: {DATA_DIR}')
else:
    print(f'✅ Phát hiện dataset tại: {DATA_DIR}')

CSV_PATH       = os.path.join(DATA_DIR, 'train.csv')
IMAGE_ZIP_PATH = os.path.join(DATA_DIR, 'train_images')
EXTRACTED_DIR  = '/content/train_images_extracted'
IMG_DIR        = os.path.join(EXTRACTED_DIR, 'train_images')

# Giải nén ảnh vào Colab local (đọc nhanh hơn Drive)
if not os.path.exists(IMG_DIR):
    if os.path.exists(IMAGE_ZIP_PATH):
        print('⏳ Giải nén train_images.zip (~1-2 phút)...')
        os.makedirs(EXTRACTED_DIR, exist_ok=True)
        !unzip -q "{IMAGE_ZIP_PATH}" -d "{EXTRACTED_DIR}"
        print('✅ Giải nén thành công!')
    else:
        print(f'❌ Không tìm thấy file zip tại {IMAGE_ZIP_PATH}')
else:
    print('✅ Thư mục ảnh đã có sẵn!')

# Kiểm tra
for name, p in [('train.csv', CSV_PATH), ('train_images/', IMG_DIR)]:
    ok = '✅' if os.path.exists(p) else '❌'
    print(f'  {ok} {name}: {p}')

## 🔧 Cell 3: Cấu hình & Chia tập dữ liệu

In [ ]:
import os, gc, json, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import faiss
from PIL import Image, UnidentifiedImageError
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from collections import defaultdict

warnings.filterwarnings('ignore')

# ─── Hyperparameters & Paths ────────────────────────────────────────────────
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED      = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# MobileCLIP
MOBILECLIP_VARIANT = 'mobileclip_s0'
MOBILECLIP_CKPT    = '/tmp/mobileclip_s0.pt'
BATCH_SIZE_CLIP    = 128
NUM_WORKERS        = 2

# YOLO
YOLO_WEIGHTS       = 'yolo11n.pt'   # pre-trained COCO; thay bằng model fine-tuned nếu có
YOLO_CONF          = 0.20
YOLO_IMGSZ         = 640
YOLO_BATCH         = 1              # stable mode; tăng nếu muốn nhanh hơn
CROP_PADDING       = 0.08           # 8% padding quanh bbox
FALLBACK_RATIO     = 0.80           # center-crop fallback khi YOLO fail

# DINOv2
DINO_MODEL         = 'dinov2_vits14'
DINO_FEAT_DIM      = 384
BATCH_SIZE_DINO    = 32             # giảm xuống 16 nếu OOM

# Retrieval
FINAL_K            = 5

# Paths
FEAT_DIR           = '/content/features'
CROP_DIR           = '/content/gallery_cropped'
RESULTS_DIR        = '/content/results'
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─── Load & Split ───────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f'📊 Dataset: {len(df):,} sản phẩm | {df["label_group"].nunique():,} nhóm')
print(df.head(2))

df_gallery = df.copy()  # toàn bộ gallery

val_idx, test_idx = train_test_split(
    df.index.tolist(), test_size=0.8, random_state=RANDOM_SEED
)
df_val  = df.loc[val_idx].reset_index(drop=True)   # 20% → tune hyperparams
df_test = df.loc[test_idx].reset_index(drop=True)  # 80% → eval cuối

print(f'\n🗂️  Gallery  : {len(df_gallery):,} ảnh')
print(f'✅ Val       : {len(df_val):,} queries → tune α, K, β')
print(f'✅ Test      : {len(df_test):,} queries → eval cuối (chạy 1 lần!)')
print(f'\n⚙️  Device   : {DEVICE}')
print(f'⚙️  YOLO     : {YOLO_WEIGHTS}')
print(f'⚙️  DINOv2   : {DINO_MODEL}')

## 🍎 Cell 4: Load MobileCLIP (hoặc CLIP Fallback)

In [ ]:
import urllib.request

if USE_MOBILECLIP:
    import mobileclip

    CKPT_URLS = {
        'mobileclip_s0': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt',
        'mobileclip_s1': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s1.pt',
    }
    if not os.path.exists(MOBILECLIP_CKPT):
        print(f'⏬ Download {MOBILECLIP_VARIANT}...')
        urllib.request.urlretrieve(CKPT_URLS[MOBILECLIP_VARIANT], MOBILECLIP_CKPT)
        print(f'✅ Saved: {MOBILECLIP_CKPT}')

    print(f'⏳ Loading {MOBILECLIP_VARIANT}...')
    clip_model, _, clip_preprocess = mobileclip.create_model_and_transforms(
        MOBILECLIP_VARIANT, pretrained=MOBILECLIP_CKPT
    )
    clip_tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
    clip_model = clip_model.to(DEVICE).eval()

    with torch.no_grad():
        _dummy = torch.randn(1, 3, 256, 256).to(DEVICE)
        CLIP_DIM = clip_model.encode_image(_dummy).shape[-1]

    MODEL_LABEL = f'MobileCLIP ({MOBILECLIP_VARIANT})'

else:
    from transformers import CLIPModel, CLIPProcessor
    HF_MODEL = 'openai/clip-vit-base-patch32'
    print(f'⏳ Loading {HF_MODEL}...')
    clip_model     = CLIPModel.from_pretrained(HF_MODEL).to(DEVICE).eval()
    clip_preprocess = CLIPProcessor.from_pretrained(HF_MODEL)
    clip_tokenizer  = None
    CLIP_DIM        = clip_model.config.projection_dim
    MODEL_LABEL     = f'CLIP ({HF_MODEL})'

print(f'\n✅ {MODEL_LABEL}')
print(f'📐 Embedding dim: {CLIP_DIM}')

## 🖼️📝 Cell 5: Trích xuất MobileCLIP Features cho Gallery

In [ ]:
# ─── Dataset helper ─────────────────────────────────────────────────────────
class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df, self.img_dir, self.transform = df, img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, self.df.iloc[idx]['image'])
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (256, 256), (128,128,128))
        if self.transform:
            return self.transform(img)
        return img


def l2_norm(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, 1e-12)


@torch.no_grad()
def extract_clip_image_features(df_in, img_dir, batch_size=BATCH_SIZE_CLIP):
    all_feats = []
    if USE_MOBILECLIP:
        ds     = ShopeeDataset(df_in, img_dir, transform=clip_preprocess)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        for imgs in tqdm(loader, desc='🖼️ CLIP image'):
            feats = clip_model.encode_image(imgs.to(DEVICE))
            all_feats.append(feats.cpu().float().numpy())
    else:
        ds = ShopeeDataset(df_in, img_dir)
        for i in tqdm(range(0, len(ds), batch_size), desc='🖼️ CLIP image'):
            batch  = [ds[j] for j in range(i, min(i+batch_size, len(ds)))]
            inputs = clip_preprocess(images=batch, return_tensors='pt',
                                     padding=True).to(DEVICE)
            feats  = clip_model.get_image_features(**inputs)
            all_feats.append(feats.cpu().float().numpy())
    return l2_norm(np.vstack(all_feats))


@torch.no_grad()
def extract_clip_text_features(df_in, batch_size=256):
    titles, all_feats = df_in['title'].fillna('').tolist(), []
    if USE_MOBILECLIP:
        for i in tqdm(range(0, len(titles), batch_size), desc='📝 CLIP text'):
            tokens = clip_tokenizer(titles[i:i+batch_size]).to(DEVICE)
            feats  = clip_model.encode_text(tokens)
            all_feats.append(feats.cpu().float().numpy())
    else:
        for i in tqdm(range(0, len(titles), batch_size), desc='📝 CLIP text'):
            inputs = clip_preprocess(
                text=titles[i:i+batch_size], return_tensors='pt',
                padding=True, truncation=True, max_length=77
            ).to(DEVICE)
            feats = clip_model.get_text_features(**inputs)
            all_feats.append(feats.cpu().float().numpy())
    return l2_norm(np.vstack(all_feats))


# ─── Cache paths ────────────────────────────────────────────────────────────
GAL_IMG_PATH = os.path.join(FEAT_DIR, 'gallery_clip_image.npy')
GAL_TXT_PATH = os.path.join(FEAT_DIR, 'gallery_clip_text.npy')

# Gallery image features
if os.path.exists(GAL_IMG_PATH) and np.load(GAL_IMG_PATH).shape[0] == len(df_gallery):
    print('✅ Load cached gallery image features')
    gallery_img_feat = np.load(GAL_IMG_PATH)
else:
    print('⚙️  Extracting gallery image features...')
    gallery_img_feat = extract_clip_image_features(df_gallery, IMG_DIR)
    np.save(GAL_IMG_PATH, gallery_img_feat)
    print(f'💾 Saved → {GAL_IMG_PATH}')

# Gallery text features
if os.path.exists(GAL_TXT_PATH) and np.load(GAL_TXT_PATH).shape[0] == len(df_gallery):
    print('✅ Load cached gallery text features')
    gallery_txt_feat = np.load(GAL_TXT_PATH)
else:
    print('⚙️  Extracting gallery text features...')
    gallery_txt_feat = extract_clip_text_features(df_gallery)
    np.save(GAL_TXT_PATH, gallery_txt_feat)
    print(f'💾 Saved → {GAL_TXT_PATH}')

print(f'\n✅ gallery_img_feat : {gallery_img_feat.shape}')
print(f'✅ gallery_txt_feat : {gallery_txt_feat.shape}')

# ─── Query features (val + test) ────────────────────────────────────────────
print('\n⚙️  Extracting val query features...')
val_img_feat = extract_clip_image_features(df_val, IMG_DIR)
val_txt_feat = extract_clip_text_features(df_val)

print('⚙️  Extracting test query features...')
test_img_feat = extract_clip_image_features(df_test, IMG_DIR)
test_txt_feat = extract_clip_text_features(df_test)

print(f'\n✅ Val  img/txt: {val_img_feat.shape} / {val_txt_feat.shape}')
print(f'✅ Test img/txt: {test_img_feat.shape} / {test_txt_feat.shape}')

## ✂️ Cell 6: YOLO Crop — Toàn bộ Gallery

In [ ]:
import math
from ultralytics import YOLO

# ─── Utility helpers (từ two_stage_pipeline.py) ─────────────────────────────
def _clamp_box(x1, y1, x2, y2, W, H):
    x1, y1 = max(0.0, x1), max(0.0, y1)
    x2, y2 = min(float(W), x2), min(float(H), y2)
    return (x1, y1, x2, y2) if (x2 > x1 and y2 > y1) else None

def _choose_best_box(boxes, scores, W, H):
    img_area = max(W * H, 1)
    cx_img, cy_img = W / 2.0, H / 2.0
    diag = math.hypot(W, H)
    best_score, best_box = -1e9, None
    for box, conf in zip(boxes, scores):
        b = _clamp_box(*box[:4], W, H)
        if b is None: continue
        x1, y1, x2, y2 = b
        ar = (x2-x1)*(y2-y1) / img_area
        if ar < 0.01 or ar > 0.95: continue
        cx, cy = (x1+x2)/2, (y1+y2)/2
        dist   = math.hypot(cx-cx_img, cy-cy_img)
        center = 1.0 - dist / max(diag, 1.0)
        area   = min(ar / 0.45, 1.0)
        score  = float(conf) + 0.20*center + 0.15*area
        if score > best_score:
            best_score, best_box = score, b
    return best_box

def _center_crop(img: Image.Image, ratio=FALLBACK_RATIO) -> Image.Image:
    w, h = img.size
    nw, nh = int(w*ratio), int(h*ratio)
    return img.crop(((w-nw)//2, (h-nh)//2, (w-nw)//2+nw, (h-nh)//2+nh))


# ─── Load YOLO ──────────────────────────────────────────────────────────────
print(f'⏳ Loading YOLO: {YOLO_WEIGHTS}')
yolo_model = YOLO(YOLO_WEIGHTS)   # downloads yolo11n.pt automatically if needed
print('✅ YOLO loaded')


# ─── Crop gallery ───────────────────────────────────────────────────────────
def crop_gallery(df_in, img_dir, save_dir, conf=YOLO_CONF, pad=CROP_PADDING):
    os.makedirs(save_dir, exist_ok=True)
    stats = {'yolo': 0, 'fallback': 0, 'error': 0}

    for _, row in tqdm(df_in.iterrows(), total=len(df_in), desc='YOLO crop'):
        fname    = row['image']
        src_path = os.path.join(img_dir, fname)
        dst_path = os.path.join(save_dir, fname)

        if os.path.exists(dst_path):   # skip if already cropped
            continue
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)

        try:
            pil = Image.open(src_path).convert('RGB')
            W, H = pil.size
            det  = yolo_model.predict(src_path, imgsz=YOLO_IMGSZ,
                                      conf=conf, verbose=False)[0]
            boxes  = det.boxes.xyxy.cpu().numpy() if det.boxes else np.zeros((0,4))
            confs  = det.boxes.conf.cpu().numpy() if det.boxes else np.zeros(0)
            best   = _choose_best_box(boxes, confs, W, H) if len(boxes) else None

            if best is not None:
                x1, y1, x2, y2 = best
                bw, bh = x2-x1, y2-y1
                x1 = max(0, x1 - bw*pad); y1 = max(0, y1 - bh*pad)
                x2 = min(W, x2 + bw*pad); y2 = min(H, y2 + bh*pad)
                cropped = pil.crop((x1, y1, x2, y2))
                stats['yolo'] += 1
            else:
                cropped = _center_crop(pil)
                stats['fallback'] += 1

        except Exception:
            try:
                pil = Image.open(src_path).convert('RGB')
                cropped = _center_crop(pil)
            except Exception:
                cropped = Image.new('RGB', (224, 224), 128)
            stats['error'] += 1

        cropped.save(dst_path)

    total = len(df_in)
    print(f'\n✅ Crop done | yolo={stats["yolo"]} ({stats["yolo"]/total:.1%}) | '
          f'fallback={stats["fallback"]} | error={stats["error"]}')
    return stats


crop_stats = crop_gallery(df_gallery, IMG_DIR, CROP_DIR)

# Kiểm tra số file crop
n_cropped = sum(1 for f in df_gallery['image'] if os.path.exists(os.path.join(CROP_DIR, f)))
print(f'📁 Crop files: {n_cropped:,} / {len(df_gallery):,}')

## 🦕 Cell 7: Trích xuất DINOv2 Features (trên YOLO crops)

In [ ]:
# ─── Giải phóng VRAM từ MobileCLIP trước khi load DINOv2 ───────────────────
# (T4 chỉ có 15 GB, không thể giữ cả hai model cùng lúc)
del clip_model
gc.collect()
torch.cuda.empty_cache()
print(f'🗑️  Freed CLIP. VRAM trống: '
      f'{torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved')

# ─── DINOv2 transform ───────────────────────────────────────────────────────
DINO_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


class CropDataset(Dataset):
    def __init__(self, df, crop_dir, transform):
        self.paths = [
            os.path.join(crop_dir, row['image'])
            for _, row in df.iterrows()
        ]
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), 128)
        return self.transform(img)


@torch.no_grad()
def extract_dino_features(df_in, crop_dir, dino_model, batch_size=BATCH_SIZE_DINO):
    ds     = CropDataset(df_in, crop_dir, DINO_TRANSFORM)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    all_feats = []
    for batch in tqdm(loader, desc='🦕 DINOv2 extract'):
        batch = batch.to(DEVICE, non_blocking=True)
        out   = dino_model.forward_features(batch)
        feats = out['x_norm_clstoken'] if isinstance(out, dict) else out
        if feats.ndim == 3: feats = feats[:, 0, :]
        feats = F.normalize(feats, dim=-1)
        all_feats.append(feats.cpu().float().numpy())
    return l2_norm(np.vstack(all_feats))


# ─── Load DINOv2 ────────────────────────────────────────────────────────────
print(f'⏳ Loading {DINO_MODEL} from torch.hub...')
dino_model = torch.hub.load('facebookresearch/dinov2', DINO_MODEL, verbose=False)
dino_model = dino_model.eval().to(DEVICE)
print('✅ DINOv2 loaded')

# ─── Gallery DINOv2 features (với cache) ────────────────────────────────────
GAL_DINO_PATH = os.path.join(FEAT_DIR, f'gallery_dino_crop_{DINO_MODEL}.npy')

if os.path.exists(GAL_DINO_PATH) and np.load(GAL_DINO_PATH).shape[0] == len(df_gallery):
    print('✅ Load cached gallery DINOv2 features')
    gallery_dino_feat = np.load(GAL_DINO_PATH)
else:
    print('⚙️  Extracting gallery DINOv2 features...')
    gallery_dino_feat = extract_dino_features(df_gallery, CROP_DIR, dino_model)
    np.save(GAL_DINO_PATH, gallery_dino_feat)
    print(f'💾 Saved → {GAL_DINO_PATH}')

# ─── Val & Test DINOv2 features ─────────────────────────────────────────────
print('\n⚙️  Extracting val DINOv2 features...')
val_dino_feat  = extract_dino_features(df_val,  CROP_DIR, dino_model)

print('⚙️  Extracting test DINOv2 features...')
test_dino_feat = extract_dino_features(df_test, CROP_DIR, dino_model)

print(f'\n✅ gallery_dino_feat : {gallery_dino_feat.shape}')
print(f'✅ val_dino_feat     : {val_dino_feat.shape}')
print(f'✅ test_dino_feat    : {test_dino_feat.shape}')

# ─── Giải phóng DINOv2, reload MobileCLIP cho bước tuning ──────────────────
del dino_model
gc.collect()
torch.cuda.empty_cache()
print('\n🗑️  DINOv2 unloaded. Reload MobileCLIP cho grid search...')

## 🔄 Cell 7b: Reload MobileCLIP sau khi extract DINOv2

In [ ]:
# Reload MobileCLIP (cần cho fuse_and_normalize và evaluate)
if USE_MOBILECLIP:
    import mobileclip
    clip_model, _, clip_preprocess = mobileclip.create_model_and_transforms(
        MOBILECLIP_VARIANT, pretrained=MOBILECLIP_CKPT
    )
    clip_tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
    clip_model = clip_model.to(DEVICE).eval()
else:
    from transformers import CLIPModel, CLIPProcessor
    clip_model      = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
    clip_preprocess = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    clip_tokenizer  = None

print(f'✅ {MODEL_LABEL} reloaded')

# ─── Core utility functions ─────────────────────────────────────────────────
def fuse_clip(img_feats, txt_feats, alpha):
    """Weighted fusion + L2 normalize."""
    fused = alpha * img_feats + (1 - alpha) * txt_feats
    return l2_norm(fused)


def build_faiss_index(features, use_gpu=True):
    idx = faiss.IndexFlatIP(features.shape[1])
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        idx = faiss.index_cpu_to_gpu(res, 0, idx)
    idx.add(np.ascontiguousarray(features.astype(np.float32)))
    return idx


def score_fusion(dino_s, clip_s, beta, top_k=FINAL_K):
    """Min-max normalize both, then blend."""
    def _mm(a): return np.clip((a - a.min())/(a.max()-a.min()+1e-8), 0, 1)
    fused = beta * _mm(dino_s.astype(np.float32)) + \
            (1-beta) * _mm(clip_s.astype(np.float32))
    k = min(top_k, len(fused))
    part  = np.argpartition(-fused, k-1)[:k]
    order = np.argsort(-fused[part])
    return part[order], fused[part[order]]


def get_gt_dict(df_in):
    gt = {}
    for _, grp in df_in.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids: gt[pid] = ids
    return gt


print('✅ Utility functions ready')

## 🔍 Cell 8: Grid Search Alpha — Stage 1 (MobileCLIP Only)

In [ ]:
import matplotlib.pyplot as plt

def evaluate_stage1(q_img, q_txt, g_img, g_txt, query_df, gallery_df, alpha, K=5):
    """Evaluate MobileCLIP-only (Stage 1)."""
    q_fused = fuse_clip(q_img, q_txt, alpha)
    g_fused = fuse_clip(g_img, g_txt, alpha)
    index   = build_faiss_index(g_fused)
    _, idxs = index.search(np.ascontiguousarray(q_fused), K+1)
    gt      = get_gt_dict(gallery_df)
    g_pids  = gallery_df['posting_id'].tolist()

    ap_list, p1_list, r5_list = [], [], []
    for i, row in enumerate(query_df.itertuples()):
        relevant = gt.get(row.posting_id, set()) - {row.posting_id}
        if not relevant: continue
        retrieved = [g_pids[j] for j in idxs[i] if g_pids[j] != row.posting_id][:K]
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, 1):
            if pid in relevant: hits += 1; ap += hits/rank
        ap_list.append(ap / min(len(relevant), K))
        p1_list.append(1.0 if retrieved and retrieved[0] in relevant else 0.0)
        r5_list.append(len(set(retrieved) & relevant) / len(relevant))
    return {'mAP@5': np.mean(ap_list), 'Precision@1': np.mean(p1_list), 'Recall@5': np.mean(r5_list)}


print('🔍 Grid Search Alpha (Stage 1 only)...')
alpha_results = []
best_alpha, best_map_alpha = 0.5, 0.0

for alpha in np.arange(0.0, 1.05, 0.1):
    alpha = round(float(alpha), 1)
    m = evaluate_stage1(val_img_feat, val_txt_feat,
                        gallery_img_feat, gallery_txt_feat,
                        df_val, df_gallery, alpha)
    alpha_results.append({'alpha': alpha, **m})
    marker = ' ← BEST' if m['mAP@5'] > best_map_alpha else ''
    print(f'  alpha={alpha:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map_alpha:
        best_map_alpha, best_alpha = m['mAP@5'], alpha

alpha_df = pd.DataFrame(alpha_results)
print(f'\n🏆 Best alpha = {best_alpha:.1f}  (val mAP@5 = {best_map_alpha:.4f})')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alpha_df['alpha'], alpha_df['mAP@5'], 'o-', color='#3B82F6', lw=2, label='mAP@5')
ax.axvline(best_alpha, ls='--', color='red', alpha=0.6, label=f'Best α={best_alpha}')
ax.set_xlabel('Alpha', fontsize=12); ax.set_ylabel('mAP@5', fontsize=12)
ax.set_title('Grid Search: Alpha (Stage 1 — MobileCLIP only)', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_alpha.png'), dpi=150)
plt.show()

## 🔍 Cell 9: Grid Search Retrieval-K (Stage 2, β=1.0)

In [ ]:
def evaluate_two_stage(q_img, q_txt, q_dino,
                        g_img, g_txt, g_dino,
                        query_df, gallery_df,
                        alpha, retrieval_k, beta, K=FINAL_K):
    """Full 2-stage evaluation on pre-computed features."""
    # Stage 1: FAISS
    q_fused = fuse_clip(q_img, q_txt, alpha)
    g_fused = fuse_clip(g_img, g_txt, alpha)
    index   = build_faiss_index(g_fused)

    gt     = get_gt_dict(gallery_df)
    g_pids = gallery_df['posting_id'].tolist()
    g_lbls = gallery_df['label_group'].values

    ap_list, p1_list, r5_list = [], [], []

    for i in tqdm(range(len(query_df)), desc=f'Eval α={alpha} K={retrieval_k} β={beta}', leave=False):
        qid      = query_df.iloc[i]['posting_id']
        relevant = gt.get(qid, set()) - {qid}
        if not relevant: continue

        # FAISS search
        q_vec = np.ascontiguousarray(q_fused[i:i+1])
        dists, idxs = index.search(q_vec, retrieval_k + 1)
        cand_idxs   = idxs[0][idxs[0] >= 0]
        clip_scores = dists[0][idxs[0] >= 0]

        if len(cand_idxs) == 0:
            ap_list.append(0.0); p1_list.append(0.0); r5_list.append(0.0)
            continue

        # Stage 2: DINOv2 rerank
        dino_scores = g_dino[cand_idxs] @ q_dino[i]   # cosine (features L2-normed)
        local_top, _ = score_fusion(dino_scores, clip_scores, beta=beta, top_k=K)
        global_top   = cand_idxs[local_top]
        retrieved    = [g_pids[j] for j in global_top]

        # Metrics
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved[:K], 1):
            if pid in relevant: hits += 1; ap += hits/rank
        ap_list.append(ap / min(len(relevant), K))
        p1_list.append(1.0 if retrieved and retrieved[0] in relevant else 0.0)
        r5_list.append(len(set(retrieved[:K]) & relevant) / len(relevant))

    return {
        'mAP@5'      : float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5'   : float(np.mean(r5_list)),
    }


# ─── Grid K (DINOv2 only, beta=1.0 để isolate effect của K) ─────────────────
print('🔍 Grid Search Retrieval-K (β=1.0, DINOv2 only)...')
k_results = []
best_k, best_map_k = 100, 0.0

for k in [50, 100, 150, 200]:
    m = evaluate_two_stage(
        val_img_feat, val_txt_feat, val_dino_feat,
        gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
        df_val, df_gallery,
        alpha=best_alpha, retrieval_k=k, beta=1.0
    )
    k_results.append({'retrieval_k': k, **m})
    marker = ' ← BEST' if m['mAP@5'] > best_map_k else ''
    print(f'  K={k:3d} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map_k:
        best_map_k, best_k = m['mAP@5'], k

k_df = pd.DataFrame(k_results)
print(f'\n🏆 Best K = {best_k}  (val mAP@5 = {best_map_k:.4f})')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(r['retrieval_k']) for r in k_results],
       [r['mAP@5'] for r in k_results], color='#10B981')
ax.set_xlabel('Retrieval K'); ax.set_ylabel('mAP@5')
ax.set_title('Grid Search: Retrieval-K (β=1.0)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_k.png'), dpi=150)
plt.show()

## 🔍 Cell 10: Grid Search Beta — Full Pipeline

In [ ]:
print('🔍 Grid Search Beta (full pipeline)...')
beta_results = []
best_beta, best_map_beta = 0.3, 0.0

for beta in np.arange(0.0, 1.05, 0.1):
    beta = round(float(beta), 1)
    m = evaluate_two_stage(
        val_img_feat, val_txt_feat, val_dino_feat,
        gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
        df_val, df_gallery,
        alpha=best_alpha, retrieval_k=best_k, beta=beta
    )
    beta_results.append({'beta': beta, **m})
    marker = ' ← BEST' if m['mAP@5'] > best_map_beta else ''
    print(f'  beta={beta:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map_beta:
        best_map_beta, best_beta = m['mAP@5'], beta

beta_df = pd.DataFrame(beta_results)
print(f'\n🏆 Best beta = {best_beta:.1f}  (val mAP@5 = {best_map_beta:.4f})')

# ─── Summary hyperparameters ────────────────────────────────────────────────
best_params = {
    'alpha'      : float(best_alpha),
    'retrieval_k': int(best_k),
    'beta'       : float(best_beta),
    'val_mAP@5'  : float(best_map_beta),
}
hp_path = os.path.join(RESULTS_DIR, 'best_hyperparams.json')
with open(hp_path, 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'\n💾 Hyperparameters saved → {hp_path}')
print(json.dumps(best_params, indent=2))

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(beta_df['beta'], beta_df['mAP@5'], 's-', color='#8B5CF6', lw=2)
ax.plot(beta_df['beta'], beta_df['Precision@1'], '^--', color='#F59E0B', lw=1.5, label='P@1')
ax.plot(beta_df['beta'], beta_df['Recall@5'],    'v--', color='#10B981', lw=1.5, label='R@5')
ax.axvline(best_beta, ls='--', color='red', alpha=0.7, label=f'Best β={best_beta}')
ax.set_xlabel('Beta (DINOv2 weight)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Grid Search: Beta — Full 2-Stage Pipeline', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_beta.png'), dpi=150)
plt.show()

## 🧪 Cell 11: Đánh giá Test Set
> ⚠️ **QUAN TRỌNG**: Chỉ chạy cell này **1 lần duy nhất** sau khi đã chốt hyperparameters!

In [ ]:
print('=' * 60)
print('🧪 FINAL TEST SET EVALUATION (chạy 1 lần!)')
print('=' * 60)
print(f'   alpha       = {best_alpha}')
print(f'   retrieval_k = {best_k}')
print(f'   beta        = {best_beta}')
print()

test_metrics = evaluate_two_stage(
    test_img_feat, test_txt_feat, test_dino_feat,
    gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
    df_test, df_gallery,
    alpha=best_alpha, retrieval_k=best_k, beta=best_beta
)

print('\n📊 FINAL RESULTS (Test Set):')
print(f'   mAP@5        = {test_metrics["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics["Recall@5"]:.4f}')

# Save
results_payload = {
    'model'      : f'{MODEL_LABEL} + DINOv2-Crop ({DINO_MODEL})',
    'yolo'       : YOLO_WEIGHTS,
    'hyperparams': best_params,
    'test_metrics': test_metrics,
    'crop_stats' : crop_stats,
}
out_path = os.path.join(RESULTS_DIR, 'final_results_twostage.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results_payload, f, indent=2, ensure_ascii=False)
print(f'\n💾 Results saved → {out_path}')

## 📊 Cell 12: So sánh với Baseline & Bảng Báo cáo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

baseline_rows = [
    {'Phương pháp': 'Baseline TF-IDF',
     'mAP@5': 0.5204, 'Precision@1': 0.5922, 'Recall@5': 0.6349},
    {'Phương pháp': 'Baseline ResNet50',
     'mAP@5': 0.5268, 'Precision@1': 0.6328, 'Recall@5': 0.5293},
    {'Phương pháp': 'Baseline MobileCLIP (Stage 1)',
     'mAP@5': 0.7708, 'Precision@1': 0.7937, 'Recall@5': 0.7430},
    {'Phương pháp': f'Tuần 4 — 2-Stage: {MODEL_LABEL} + YOLO-DINOv2',
     'mAP@5'      : test_metrics['mAP@5'],
     'Precision@1': test_metrics['Precision@1'],
     'Recall@5'   : test_metrics['Recall@5']},
]

report_df = pd.DataFrame(baseline_rows)
metric_cols = ['mAP@5', 'Precision@1', 'Recall@5']
report_df[metric_cols] = report_df[metric_cols].round(4)

# Lưu CSV
report_csv = os.path.join(RESULTS_DIR, 'REPORT_twostage_comparison.csv')
report_df.to_csv(report_csv, index=False, encoding='utf-8-sig')

print('📊 Bảng so sánh:')
display(report_df.style
    .highlight_max(subset=metric_cols, color='#86EFAC')
    .format({c: '{:.4f}' for c in metric_cols}))

# Tính delta so với MobileCLIP baseline
base = report_df[report_df['Phương pháp'].str.contains('Stage 1')].iloc[0]
new  = report_df.iloc[-1]
print('\n🎯 Cải thiện so với MobileCLIP Baseline:')
for col in metric_cols:
    delta = new[col] - base[col]
    sign  = '+' if delta >= 0 else ''
    print(f'   {col:15s}: {base[col]:.4f} → {new[col]:.4f}  ({sign}{delta:.4f} = {sign}{delta*100:.2f}%)')

# ─── Bar chart ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#94A3B8', '#94A3B8', '#3B82F6', '#10B981']

for ax, metric in zip(axes, metric_cols):
    vals   = report_df[metric].tolist()
    labels = [p.split('—')[-1].strip() if '—' in p else p
              for p in report_df['Phương pháp']]
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='white', width=0.6)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('So sánh các phương pháp — Shopee Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n💾 Saved → {report_csv}')

## 🖼️ Cell 13: Visualize Kết quả Truy vấn Top-5

In [ ]:
def visualize_query(query_idx, df_query, df_gallery,
                     q_img_feat, q_txt_feat, q_dino_feat,
                     g_img_feat, g_txt_feat, g_dino_feat,
                     img_dir, crop_dir,
                     alpha, retrieval_k, beta, K=5, show_crop=True):
    """Hiển thị query và top-5 retrieved ảnh kèm nhãn TP/FP."""

    row   = df_query.iloc[query_idx]
    q_lbl = row['label_group']

    # Run pipeline
    q_fused = fuse_clip(q_img_feat[query_idx:query_idx+1],
                         q_txt_feat[query_idx:query_idx+1], alpha)
    g_fused = fuse_clip(g_img_feat, g_txt_feat, alpha)
    index   = build_faiss_index(g_fused)
    dists, idxs = index.search(np.ascontiguousarray(q_fused), retrieval_k+1)

    cand     = idxs[0][idxs[0] >= 0]
    c_dists  = dists[0][idxs[0] >= 0]
    d_scores = g_dino_feat[cand] @ q_dino_feat[query_idx]
    local_top, scores = score_fusion(d_scores, c_dists, beta=beta, top_k=K)
    top_global = cand[local_top]

    # Plot
    fig, axes = plt.subplots(2 if show_crop else 1, K+1,
                             figsize=(3*(K+1), 3*(2 if show_crop else 1)+1))
    if not show_crop: axes = [axes]

    def _load(path):
        try: return Image.open(path).convert('RGB')
        except: return Image.new('RGB', (224, 224), 200)

    for row_i, (ax_row, use_crop) in enumerate(zip(axes, [False, True] if show_crop else [False])):
        d = crop_dir if use_crop else img_dir
        label = '(crop)' if use_crop else '(orig)'

        q_img = _load(os.path.join(d, row['image']))
        ax_row[0].imshow(q_img)
        ax_row[0].set_title(f'QUERY {label}\n{str(q_lbl)[:12]}', fontsize=8, color='navy')
        ax_row[0].axis('off')

        for rank, (g_idx, sc) in enumerate(zip(top_global, scores), 1):
            g_row  = df_gallery.iloc[g_idx]
            g_img  = _load(os.path.join(d, g_row['image']))
            is_tp  = g_row['label_group'] == q_lbl
            color  = '#16A34A' if is_tp else '#DC2626'
            tag    = '✅ TP' if is_tp else '❌ FP'
            ax_row[rank].imshow(g_img)
            ax_row[rank].set_title(f'Rank {rank} {tag}\n{sc:.3f}', fontsize=8, color=color)
            ax_row[rank].axis('off')
            for spine in ax_row[rank].spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(3)

    title = (f'Query #{query_idx} | α={alpha} K={retrieval_k} β={beta} | '
             f'Title: {row["title"][:50]}')
    fig.suptitle(title, fontsize=9)
    plt.tight_layout()
    plt.show()


# ─── Visualize 3 queries ngẫu nhiên ─────────────────────────────────────────
rng = np.random.default_rng(42)
sample_idxs = rng.choice(len(df_test), size=3, replace=False)

for q_idx in sample_idxs:
    visualize_query(
        int(q_idx), df_test, df_gallery,
        test_img_feat, test_txt_feat, test_dino_feat,
        gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
        IMG_DIR, CROP_DIR,
        alpha=best_alpha, retrieval_k=best_k, beta=best_beta,
        show_crop=True
    )

## 🔍 Cell 14: Demo Single Query (Production-style)

In [ ]:
# ─── Demo: Truy vấn 1 ảnh bất kỳ ────────────────────────────────────────────
DEMO_IDX   = 0                        # Thay đổi index này để xem query khác
DEMO_ROW   = df_test.iloc[DEMO_IDX]
DEMO_PATH  = os.path.join(IMG_DIR, DEMO_ROW['image'])
DEMO_TITLE = DEMO_ROW['title']

print(f'🔍 Demo Query:')
print(f'   Image : {DEMO_PATH}')
print(f'   Title : {DEMO_TITLE}')
print(f'   Label : {DEMO_ROW["label_group"]}')

t0 = time.perf_counter()

# --- Stage 1: MobileCLIP ---
q_img_vec = test_img_feat[DEMO_IDX:DEMO_IDX+1]
q_txt_vec = test_txt_feat[DEMO_IDX:DEMO_IDX+1]
q_fused   = fuse_clip(q_img_vec, q_txt_vec, best_alpha)
g_fused   = fuse_clip(gallery_img_feat, gallery_txt_feat, best_alpha)
idx       = build_faiss_index(g_fused)
t_faiss   = time.perf_counter()
dists, idxs = idx.search(np.ascontiguousarray(q_fused), best_k + 1)
t_faiss   = time.perf_counter() - t_faiss

cand      = idxs[0][idxs[0] >= 0]
c_dists   = dists[0][idxs[0] >= 0]

# --- Stage 2: DINOv2 ---
t_dino   = time.perf_counter()
d_scores = gallery_dino_feat[cand] @ test_dino_feat[DEMO_IDX]
local_top, final_scores = score_fusion(d_scores, c_dists, best_beta, top_k=5)
top5     = cand[local_top]
t_dino   = time.perf_counter() - t_dino
t_total  = time.perf_counter() - t0

print(f'\n⏱️  Latency breakdown:')
print(f'   FAISS search : {t_faiss*1000:.1f} ms')
print(f'   DINOv2 rerank: {t_dino*1000:.1f} ms')
print(f'   Total        : {t_total*1000:.1f} ms')

print(f'\n🏆 Top-5 Results:')
for rank, (g_idx, sc) in enumerate(zip(top5, final_scores), 1):
    g_row  = df_gallery.iloc[g_idx]
    is_tp  = g_row['label_group'] == DEMO_ROW['label_group']
    tag    = '✅ TP' if is_tp else '❌ FP'
    print(f'   Rank {rank} {tag} | score={sc:.4f} | {g_row["title"][:60]}')

# Show images
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
def _load(p):
    try: return Image.open(p).convert('RGB')
    except: return Image.new('RGB', (224,224), 200)

axes[0].imshow(_load(DEMO_PATH))
axes[0].set_title(f'QUERY\n{DEMO_TITLE[:20]}', fontsize=8, color='navy')
axes[0].axis('off')

for rank, (g_idx, sc) in enumerate(zip(top5, final_scores), 1):
    g_row = df_gallery.iloc[g_idx]
    gpath = os.path.join(IMG_DIR, g_row['image'])
    is_tp = g_row['label_group'] == DEMO_ROW['label_group']
    c     = '#16A34A' if is_tp else '#DC2626'
    axes[rank].imshow(_load(gpath))
    axes[rank].set_title(f'Rank {rank}\n{"✅ TP" if is_tp else "❌ FP"} {sc:.3f}',
                         fontsize=8, color=c)
    axes[rank].axis('off')
    for sp in axes[rank].spines.values(): sp.set_edgecolor(c); sp.set_linewidth(3)

plt.suptitle(f'Demo Query: "{DEMO_TITLE[:70]}"', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'demo_query.png'), dpi=150)
plt.show()

---
## ✅ Pipeline hoàn tất!

### Files đã tạo trong `/content/results/`:
| File | Nội dung |
|------|----------|
| `best_hyperparams.json` | α, K, β tối ưu |
| `final_results_twostage.json` | Metrics trên test set |
| `REPORT_twostage_comparison.csv` | Bảng so sánh với baselines |
| `grid_alpha.png` | Biểu đồ grid search α |
| `grid_k.png` | Biểu đồ grid search K |
| `grid_beta.png` | Biểu đồ grid search β |
| `comparison_chart.png` | Bar chart so sánh tất cả methods |
| `demo_query.png` | Ảnh demo truy vấn |

### Files features trong `/content/features/`:
| File | Shape |
|------|-------|
| `gallery_clip_image.npy` | (34250, 512) |
| `gallery_clip_text.npy` | (34250, 512) |
| `gallery_dino_crop_dinov2_vits14.npy` | (34250, 384) |

> 💡 **Tip**: Copy `/content/features/` và `/content/results/` lên Google Drive để không mất khi runtime reset!